# WaterSoftHack 2026 — Day 1: Cloud Computing (Hands-On)

**Welcome!** You are looking at a Jupyter notebook, but the computer running it is **not your laptop**.
It is a virtual machine on **Jetstream2**, a research cloud in Indiana, paid for with our project's
NSF **ACCESS** credits. You reached it with nothing but a browser and a login. That is cloud computing,
and by the end of this hour you will have used it for real water science.

## The story we are building (all three days)

We are building a **real-time flood early-warning system** for a network of USGS stream gauges.

- **Today (Cloud):** pull *live* streamflow data, train a short-term forecast, then scale it to many
  gauges at once.
- **Tomorrow (Edge):** move the smarts out to the sensors themselves, under real field limits.
- **Thursday (Hybrid):** combine the two into one pipeline.

## What you will do in the next hour

1. See exactly where your code is running (spoiler: the cloud).
2. Pull **live** river data from the USGS national gauge network.
3. Store it the way cloud systems do.
4. Train a **streamflow now-cast** that predicts the next hour of flow.
5. **Scale out** to many gauges at once and watch the cloud's real superpower kick in.
6. Learn how to *not* waste our shared credits.

> No prior cloud experience needed. Run each cell with **Shift+Enter**. Read the short notes between cells.

---
## Part 0 — Wait, where is this code actually running?

Your browser is on your laptop. But the **kernel** (the thing that runs the Python) lives on a
remote machine in the cloud. Run the cell below and look at the answers. The hostname, the number of
CPU cores, the operating system: none of that is your laptop. It is a computer you were handed, on
demand, over the internet.

**Think about it.** You did not buy this machine, install an OS, or plug in any RAM. You asked for a
computer and got one. When class ends, we hand it back. *That* is the cloud idea: compute is a utility
you rent by the hour, like electricity, not a box you own.

---
## Part 1 — Pull live water data from the cloud

The USGS runs a national network of stream gauges and publishes their readings through a free web API
(the *National Water Information System*, NWIS). We will ask it for **discharge** (river flow, USGS
parameter code `00060`, in cubic feet per second) at the **Iowa River at Iowa City** gauge — a few
minutes from campus.

The helper below fetches real data. If the network hiccups during class, it **falls back to a
realistic synthetic gauge** so nothing breaks. Either way the rest of the notebook works.

That curve is a **hydrograph**. Every bump is real water moving down a real river, delivered to a
computer in Indiana, and drawn on your screen — with about ten lines of code and zero infrastructure
of your own. Try changing `IOWA_CITY` to another 8-digit USGS site number later and re-run.

---
## Part 2 — Where does the data live? Cloud storage

On your laptop you save files to a hard drive. The cloud gives you two flavors of storage:

- **Block storage** — behaves like a normal disk attached to your VM (what your home folder uses here).
- **Object storage** — a giant web-accessible bucket of files (Amazon calls it *S3*; OpenStack calls it
  *Swift*). This is how huge public datasets are shared. For example, the entire **National Water Model**
  forecast archive lives in public cloud object storage that anyone can read.

For now we will save our data to the VM's disk, then read it back — the everyday version of the same idea.

---
## Part 3 — Train a streamflow *now-cast*

A **now-cast** predicts the very near future — here, **next hour's flow** from the last several hours.
It is the core of a flood early-warning system: if the model says flow is about to spike, you warn people.

One subtlety makes the difference between a model that works and one that embarrasses you. River flow
trends up and down over weeks, and a tree-based model cannot predict a value it never saw in training.
So instead of predicting the flow *level* directly, we predict the **hourly change** and add it back to
the current flow. The change is stationary and well-behaved, which makes the forecast robust.

The recipe:

- **Features:** current flow, the last 6 hourly values (`lag1`..`lag6`), a 6-hour average, and hour of day.
- **Target:** the **change** in flow over the next hour (then reconstruct the level).
- **Split by time:** train on the earlier 70%, test on the most recent 30% (never shuffle a time series).
- **Model:** a Random Forest — robust and needs no tuning to work.
- **Scores:** **RMSE** (typical error, in cfs) and **NSE** (Nash-Sutcliffe Efficiency — the hydrologist's
  favorite: 1.0 is perfect, 0 means "no better than predicting the average").

You just trained a machine-learning model **on cloud hardware**. It never touched your laptop. And if this
model were bigger, say a deep neural network, you could shut down this plain CPU machine and boot one with
an **NVIDIA A100 GPU** on Jetstream2 in a couple of minutes, for a few more credits per hour. Renting the
*right* hardware for the moment, then giving it back, is the whole game.

---
## Part 4 — Scale out: the cloud's real superpower

One gauge was easy. But a real flood system watches **thousands** of gauges. The National Water Model
forecasts **2.7 million** river reaches. You cannot do that one at a time.

First we train a now-cast for **ten real Iowa-basin gauges** at once. Then we simulate a much larger
network to see the cloud's defining trick: **elasticity** — ask for many CPU cores and finish in a
fraction of the time.

Ten rivers, ten forecasts, most with a solid NSE. Now let's feel **scale**. We simulate a larger network
of gauges and train a now-cast for every one — first **sequentially** on one core, then **in parallel**
across every core in this VM.

**That gap is the point.** Same code, same machine, but by using all the cores at once the job finished
several times faster. And on Jetstream2 you are not stuck with this VM's cores: you could launch a
64-core machine for one hour (about 64 credits), forecast an entire state's gauge network before lunch,
then shelve it. Compute expands to fit the problem and shrinks back when you are done.

---
## Part 5 — Be a good cloud citizen (this is real money)

Our project has **200,000 ACCESS credits**, and on Jetstream2 **1 credit = 1 CPU core running for 1 hour**.
This whole workshop costs about a hundred credits. But credits are shared and real, so three habits matter:

- **Shelve your VM when you are done.** A *running* VM costs credits every hour; a *shelved* one costs **zero**.
  Leaving a 32-core VM on overnight by accident burns ~350 credits for nothing.
- **Right-size the machine.** Do not rent 64 cores to edit text. Match the machine to the task.
- **Ask for a GPU only when you will use it.** GPUs cost more per hour. Boot one, do the training, shelve it.

Cloud computing is renting. The bill is the resources you leave running, not the code you write.

---
## Wrap-up

In one hour, entirely on the cloud, you:

- ran code on a rented computer in another state,
- pulled **live** data from the national river-gauge network,
- stored and reloaded it the cloud way,
- trained a **streamflow now-cast** and scored it with RMSE and NSE,
- **scaled out** across gauges and measured the speedup,
- and learned how to keep the bill sane.

### Looking ahead to tomorrow (Edge)

Notice what we did today: we shipped **every raw reading** to the cloud and did all the thinking there.
That is fine when the gauge sits next to fiber internet. But most gauges are in remote places, running on a
solar panel and a weak cell signal. Sending everything is slow, expensive, and dies when the signal drops.

**Tomorrow we flip it around:** put a little bit of intelligence *inside the sensor* so it decides what is
worth sending. That is **edge computing**, and we will simulate a whole fleet of smart gauges.

---
## Optional challenges (if you finish early)

1. **Different river.** Change `IOWA_CITY` to a USGS gauge near your own university
   (find one at https://waterdata.usgs.gov/) and re-run Parts 1–3.
2. **Longer horizon.** In `make_features`, call it with `horizon=3` (3 hours ahead). Does NSE drop?
   Why is forecasting further out harder?
3. **Different model.** Swap `RandomForestRegressor` for
   `from sklearn.ensemble import GradientBoostingRegressor`. Compare NSE.
4. **More history.** Increase `n_lags` to 12. Does more past help the forecast?